# Lab — Learning visual representations without labels

**Scenario.** An industrial organization has more than 100,000 historical inspection images, but only a small labelled subset. It wants a reusable domain encoder for classification, retrieval, and future dense tasks.

This notebook uses a deterministic, CPU-scale proxy of that archive. Labels are generated so we can evaluate representations, but the contrastive and masked-pretraining loops never receive them. The measured values teach an evaluation protocol; they are not production benchmarks or reproductions of SimCLR, DINO, MAE, DINOv2, or DINOv3.

![Architecture choices and learning objectives shown as separate but connected decisions.](assets/architecture-objective-shift.svg)


## 0. Experiment contract

We will:

- implement NT-Xent and expose every matrix dimension;
- compare weak, domain-valid, and domain-invalid augmentation policies;
- train only small educational encoders on a bounded corpus;
- demonstrate EMA teacher, centering, sharpening, and masked-patch mechanics;
- compare random, official ImageNet-supervised, and domain-SSL representations;
- use a factory-held-out test source and stratified label budgets;
- evaluate probes, k-NN, retrieval, geometry, collapse, and source bias; and
- save evidence with explicit limitations.

Default execution is credential-free and network-free except for the same official torchvision weight download convention used in Courses 01–03. Official DINOv2 loading is opt-in because it downloads versioned source and a checkpoint through `torch.hub`.


In [ ]:
from __future__ import annotations

import copy
import json
import math
import os
import platform
import random
import time
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import adjusted_rand_score, f1_score, recall_score, silhouette_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, normalize
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import ResNet18_Weights, resnet18

SEED = 17
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

CPU_SAFE = os.getenv("CV_FULL_RUN", "0") != "1"
if CPU_SAFE:
    DEVICE = torch.device("cpu")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

ARTIFACT_DIR = Path(".artifacts/self_supervised_representation_learning")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

environment = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "scikit_learn": sklearn.__version__,
    "device": str(DEVICE),
    "cpu_safe": CPU_SAFE,
    "seed": SEED,
}
print(json.dumps(environment, indent=2))


## 1. Generate a source-aware inspection corpus

Each source has a different camera/background signature. Each semantic class has controlled evidence: scratch geometry, a dent, coloured contamination, or a missing edge. This lets us ask whether an embedding organizes by component state or by factory appearance.

The SSL dataset returns only two transformed image tensors. Labels and source IDs remain outside that training interface and are used later for evaluation.


In [ ]:
CLASS_NAMES = ["Normal", "Scratch", "Dent", "Contamination", "Missing Edge"]
SOURCE_COLORS = {
    "Factory A": (42, 48, 54),
    "Factory B": (45, 50, 55),
    "Factory C": (48, 52, 57),
}


@dataclass(frozen=True)
class Sample:
    sample_id: str
    image: Image.Image
    label: int
    source: str
    defect_size: str


def make_inspection_image(label: int, source: str, sample_seed: int, size: int = 64) -> tuple[Image.Image, str]:
    rng = np.random.default_rng(sample_seed)
    base = np.full((size, size, 3), SOURCE_COLORS[source], dtype=np.float32)
    gradient = np.linspace(-7, 9, size, dtype=np.float32)[None, :, None]
    base += gradient + rng.normal(0, 2.2, base.shape)
    image = Image.fromarray(np.uint8(np.clip(base, 0, 255)), mode="RGB")
    draw = ImageDraw.Draw(image)

    cx, cy = 32 + int(rng.integers(-3, 4)), 32 + int(rng.integers(-3, 4))
    radius = 21 + int(rng.integers(-2, 3))
    metal = tuple(int(v) for v in rng.integers(145, 181, size=3))
    draw.ellipse((cx - radius, cy - radius, cx + radius, cy + radius), fill=metal, outline=(220, 225, 230), width=2)
    draw.ellipse((cx - 7, cy - 7, cx + 7, cy + 7), fill=(62, 67, 73), outline=(205, 210, 215), width=2)

    defect_size = "none"
    if label == 1:
        length = int(rng.integers(13, 25))
        y = cy + int(rng.integers(-10, 11))
        draw.line((cx - length // 2, y, cx + length // 2, y + int(rng.integers(-3, 4))), fill=(35, 35, 38), width=2)
        defect_size = "small" if length < 18 else "large"
    elif label == 2:
        r = int(rng.integers(4, 8))
        x, y = cx + int(rng.integers(-9, 10)), cy + int(rng.integers(-9, 10))
        draw.ellipse((x - r, y - r, x + r, y + r), fill=(105, 111, 118), outline=(85, 90, 96), width=1)
        defect_size = "small" if r <= 5 else "large"
    elif label == 3:
        r = int(rng.integers(4, 8))
        x, y = cx + int(rng.integers(-10, 11)), cy + int(rng.integers(-10, 11))
        contamination = (182, 55, 42) if sample_seed % 2 else (50, 145, 71)
        draw.ellipse((x - r, y - r, x + r, y + r), fill=contamination)
        defect_size = "small" if r <= 5 else "large"
    elif label == 4:
        width = int(rng.integers(7, 13))
        draw.rectangle((cx + radius - width, cy - 5, cx + radius + 2, cy + 5), fill=SOURCE_COLORS[source])
        defect_size = "small" if width < 10 else "large"

    return image.filter(ImageFilter.GaussianBlur(radius=0.25)), defect_size


samples = []
per_class = {"Factory A": 50, "Factory B": 50, "Factory C": 30}
for source_index, (source, count) in enumerate(per_class.items()):
    for label in range(len(CLASS_NAMES)):
        for item in range(count):
            sample_seed = SEED * 10_000 + source_index * 1_000 + label * 100 + item
            image, defect_size = make_inspection_image(label, source, sample_seed)
            samples.append(Sample(f"{source_index}-{label}-{item:03d}", image, label, source, defect_size))

metadata = pd.DataFrame({
    "sample_id": [s.sample_id for s in samples],
    "label": [s.label for s in samples],
    "class_name": [CLASS_NAMES[s.label] for s in samples],
    "source": [s.source for s in samples],
    "defect_size": [s.defect_size for s in samples],
})
train_indices = metadata.index[metadata.source.isin(["Factory A", "Factory B"])].to_numpy()
test_indices = metadata.index[metadata.source.eq("Factory C")].to_numpy()
assert len(train_indices) == 500 and len(test_indices) == 150
print(metadata.groupby(["source", "class_name"]).size().unstack(fill_value=0))


In [ ]:
fig, axes = plt.subplots(3, len(CLASS_NAMES), figsize=(12, 7))
for row, source in enumerate(SOURCE_COLORS):
    for col, class_name in enumerate(CLASS_NAMES):
        index = metadata.index[(metadata.source == source) & (metadata.class_name == class_name)][0]
        axes[row, col].imshow(samples[index].image)
        axes[row, col].set_title(class_name if row == 0 else "", fontsize=9)
        axes[row, col].set_ylabel(source, fontsize=9)
        axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
plt.suptitle("Same semantic states under three source signatures")
plt.tight_layout()
plt.show()


## 2. Augmentation is part of the objective

Three policies answer different questions:

- **weak:** nearly identical views; easy agreement but limited invariance pressure;
- **domain-valid:** bounded crop, brightness, contrast, blur, and flip;
- **domain-invalid:** aggressive crop plus grayscale, which can erase colour evidence required for contamination.

We inspect the views before training. A production system would add automated retained-evidence checks and domain-expert review.


In [ ]:
POLICIES = {
    "weak": transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.RandomHorizontalFlip(p=0.2),
        transforms.ToTensor(),
    ]),
    "domain_valid": transforms.Compose([
        transforms.RandomResizedCrop(64, scale=(0.84, 1.0), ratio=(0.94, 1.06), antialias=True),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.05, hue=0.02),
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5)),
        transforms.ToTensor(),
    ]),
    "domain_invalid": transforms.Compose([
        transforms.RandomResizedCrop(64, scale=(0.48, 1.0), ratio=(0.8, 1.2), antialias=True),
        transforms.RandomGrayscale(p=1.0),
        transforms.ColorJitter(brightness=0.45, contrast=0.45),
        transforms.ToTensor(),
    ]),
}


class UnlabeledPairs(Dataset):
    def __init__(self, sample_rows, transform):
        self.sample_rows = list(sample_rows)
        self.transform = transform

    def __len__(self):
        return len(self.sample_rows)

    def __getitem__(self, index):
        image = self.sample_rows[index].image
        return self.transform(image), self.transform(image)


contamination = next(s for s in samples if s.source == "Factory A" and s.label == 3 and s.defect_size == "small")
fig, axes = plt.subplots(len(POLICIES), 3, figsize=(8, 7))
to_pil = transforms.ToPILImage()
for row, (name, policy) in enumerate(POLICIES.items()):
    axes[row, 0].imshow(contamination.image)
    axes[row, 0].set_ylabel(name.replace("_", " "), fontsize=10)
    axes[row, 1].imshow(to_pil(policy(contamination.image)))
    axes[row, 2].imshow(to_pil(policy(contamination.image)))
    for ax in axes[row]: ax.set_xticks([]); ax.set_yticks([])
axes[0, 0].set_title("original"); axes[0, 1].set_title("view A"); axes[0, 2].set_title("view B")
plt.suptitle("The invalid policy can remove label-relevant colour")
plt.tight_layout(); plt.show()


## 3. Implement NT-Xent manually

For batch size $B$ and embedding width $D$, concatenate two view batches into a $2B	imes D$ matrix. After normalization, the pairwise cosine matrix is $2B	imes2B$. Mask the diagonal and point each row to its positive partner in the other half.

![Contrastive learning turns two augmented views into a positive pair and uses other views as negatives.](assets/contrastive-learning.svg)


In [ ]:
def ntxent_loss(z1: torch.Tensor, z2: torch.Tensor, temperature: float = 0.1, return_trace: bool = False):
    if z1.shape != z2.shape or z1.ndim != 2:
        raise ValueError("z1 and z2 must both have shape [batch, embedding]")
    batch = z1.shape[0]
    embeddings = F.normalize(torch.cat([z1, z2], dim=0), dim=1)
    cosine = embeddings @ embeddings.T
    logits = cosine / temperature
    diagonal = torch.eye(2 * batch, dtype=torch.bool, device=logits.device)
    logits = logits.masked_fill(diagonal, -torch.inf)
    targets = (torch.arange(2 * batch, device=logits.device) + batch) % (2 * batch)
    loss = F.cross_entropy(logits, targets)

    explicit = -(logits[torch.arange(2 * batch), targets] - torch.logsumexp(logits, dim=1)).mean()
    torch.testing.assert_close(loss, explicit)
    if return_trace:
        probabilities = torch.softmax(logits, dim=1)
        return loss, {
            "embeddings": embeddings,
            "cosine": cosine,
            "scaled_logits": logits,
            "positive_indices": targets,
            "probabilities": probabilities,
        }
    return loss


z1 = torch.tensor([[1.0, 0.0, 0.2], [0.0, 1.0, 0.1], [0.7, 0.4, 0.2]])
z2 = torch.tensor([[0.9, 0.1, 0.2], [0.1, 0.9, 0.2], [0.6, 0.5, 0.1]])
example_loss, trace = ntxent_loss(z1, z2, temperature=0.1, return_trace=True)
print("embedding matrix [2B, D]:", tuple(trace["embeddings"].shape))
print(trace["embeddings"].round(decimals=3))
print()
print("similarity matrix [2B, 2B]:", tuple(trace["cosine"].shape))
print(trace["cosine"].round(decimals=3))
print()
print("positive column per row:", trace["positive_indices"].tolist())
print("positive probabilities:", trace["probabilities"][torch.arange(6), trace["positive_indices"]].round(decimals=3).tolist())
print("NT-Xent:", round(example_loss.item(), 4))


### Temperature experiment

Holding similarities fixed isolates the effect of $	au$. A small value makes the row distribution sharp; a large value spreads probability across alternatives. This changes both confidence and gradients.


In [ ]:
temperatures = [0.05, 0.1, 0.5, 1.0]
anchor_similarities = torch.tensor([0.82, 0.71, 0.35, 0.12, -0.05])
temperature_rows = []
fig, ax = plt.subplots(figsize=(8, 4))
for temperature in temperatures:
    probabilities = torch.softmax(anchor_similarities / temperature, dim=0).numpy()
    temperature_rows.append({"temperature": temperature, "positive_probability": probabilities[0], "entropy": float(-(probabilities * np.log(probabilities + 1e-12)).sum())})
    ax.plot(range(len(probabilities)), probabilities, marker="o", label=f"τ={temperature}")
ax.set(xlabel="candidate index (0 is positive)", ylabel="softmax probability", title="Temperature changes similarity geometry")
ax.legend(); ax.grid(alpha=0.25); plt.show()
temperature_results = pd.DataFrame(temperature_rows)
temperature_results


## 4. Train a tiny SimCLR encoder under three policies

The encoder architecture, optimizer, batch size, seed, and training budget stay fixed. Only the view policy changes. The projection head receives NT-Xent gradients, while downstream evaluation later uses the encoder output before projection.

This tiny CPU experiment demonstrates mechanics. It is not a reproduction of published SimCLR accuracy or training scale.


In [ ]:
class TinyEncoder(nn.Module):
    def __init__(self, feature_dim: int = 64):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, stride=2, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.embedding = nn.Linear(64, feature_dim)

    def forward_feature_map(self, x):
        return self.features(x)

    def forward(self, x):
        feature_map = self.forward_feature_map(x)
        return self.embedding(self.pool(feature_map).flatten(1))


class ProjectionHead(nn.Module):
    def __init__(self, input_dim: int = 64, projection_dim: int = 32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Linear(128, projection_dim)
        )

    def forward(self, x):
        return self.net(x)


unlabeled_train_samples = [samples[index] for index in train_indices]


def train_simclr(policy_name: str, epochs: int = 12):
    torch.manual_seed(SEED)
    encoder = TinyEncoder().to(DEVICE)
    projector = ProjectionHead().to(DEVICE)
    optimizer = torch.optim.AdamW(list(encoder.parameters()) + list(projector.parameters()), lr=1e-3, weight_decay=1e-4)
    generator = torch.Generator().manual_seed(SEED)
    loader = DataLoader(
        UnlabeledPairs(unlabeled_train_samples, POLICIES[policy_name]),
        batch_size=64,
        shuffle=True,
        num_workers=0,
        drop_last=True,
        generator=generator,
    )
    history = []
    started = time.perf_counter()
    for epoch in range(epochs):
        encoder.train(); projector.train()
        losses = []
        for view_a, view_b in loader:
            view_a, view_b = view_a.to(DEVICE), view_b.to(DEVICE)
            loss = ntxent_loss(projector(encoder(view_a)), projector(encoder(view_b)), temperature=0.15)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        history.append({"policy": policy_name, "epoch": epoch + 1, "loss": float(np.mean(losses))})
    elapsed = time.perf_counter() - started
    return encoder.cpu().eval(), pd.DataFrame(history), elapsed


ssl_encoders, histories, training_times = {}, [], {}
for policy_name in POLICIES:
    encoder, history, elapsed = train_simclr(policy_name)
    ssl_encoders[policy_name] = encoder
    histories.append(history)
    training_times[policy_name] = elapsed
    print(f"{policy_name:>14}: final loss={history.loss.iloc[-1]:.4f}, seconds={elapsed:.1f}")

ssl_history = pd.concat(histories, ignore_index=True)
fig, ax = plt.subplots(figsize=(7, 4))
for name, group in ssl_history.groupby("policy"):
    ax.plot(group.epoch, group.loss, marker="o", label=name.replace("_", " "))
ax.set(xlabel="epoch", ylabel="NT-Xent", title="Pretraining loss is observable—but not downstream evidence")
ax.legend(); ax.grid(alpha=0.25); plt.show()


## 5. Teacher–student mechanics: stop-gradient, EMA, centering, sharpening

BYOL and DINO are not the same algorithm, but both make an asymmetric online/student path learn from a target/teacher path. The next cell demonstrates three primitives without claiming to reproduce either system:

1. the teacher target is detached;
2. teacher weights receive an EMA update rather than optimizer gradients; and
3. centering and temperature alter the entropy of teacher distributions.

![Student and teacher paths connected by a self-distillation loss and slow EMA weight update.](assets/teacher-student-learning.svg)


In [ ]:
torch.manual_seed(SEED)
student = nn.Linear(8, 6, bias=False)
teacher = copy.deepcopy(student)
predictor = nn.Linear(6, 6)
optimizer = torch.optim.SGD(list(student.parameters()) + list(predictor.parameters()), lr=0.2)
view_a = torch.randn(12, 8)
view_b = view_a + 0.15 * torch.randn(12, 8)

with torch.no_grad():
    target_before = F.normalize(teacher(view_b), dim=1)
prediction = F.normalize(predictor(student(view_a)), dim=1)
teacher_student_loss = 2 - 2 * (prediction * target_before).sum(dim=1).mean()
optimizer.zero_grad(); teacher_student_loss.backward(); optimizer.step()
assert all(parameter.grad is None for parameter in teacher.parameters())

distance_before_ema = torch.linalg.vector_norm(student.weight.detach() - teacher.weight.detach()).item()
momentum = 0.96
with torch.no_grad():
    for teacher_parameter, student_parameter in zip(teacher.parameters(), student.parameters()):
        teacher_parameter.mul_(momentum).add_(student_parameter, alpha=1 - momentum)
distance_after_ema = torch.linalg.vector_norm(student.weight.detach() - teacher.weight.detach()).item()

teacher_logits = torch.tensor([[3.2, 1.1, 0.4, -0.2], [2.8, 1.5, 0.2, -0.4], [3.5, 0.8, 0.5, -0.1]])
running_center = teacher_logits.mean(dim=0, keepdim=True)

def distribution_entropy(logits, temperature):
    probabilities = torch.softmax(logits / temperature, dim=1)
    return probabilities, float((-(probabilities * probabilities.clamp_min(1e-12).log()).sum(dim=1)).mean())

raw_probabilities, raw_entropy = distribution_entropy(teacher_logits, 0.5)
centered_probabilities, centered_entropy = distribution_entropy(teacher_logits - running_center, 0.5)
sharpened_probabilities, sharpened_entropy = distribution_entropy(teacher_logits - running_center, 0.1)

teacher_trace = pd.DataFrame([
    {"signal": "student_teacher_loss", "value": teacher_student_loss.item()},
    {"signal": "weight_distance_before_ema", "value": distance_before_ema},
    {"signal": "weight_distance_after_ema", "value": distance_after_ema},
    {"signal": "raw_teacher_entropy", "value": raw_entropy},
    {"signal": "centered_teacher_entropy", "value": centered_entropy},
    {"signal": "centered_sharpened_entropy", "value": sharpened_entropy},
])
assert distance_after_ema < distance_before_ema
teacher_trace


**Interpretation.** EMA moves the teacher toward the updated student but does not make them identical. Centering prevents a persistent dominant logit from controlling every target. Lower teacher temperature sharpens the target distribution. Collapse avoidance still depends on the full objective, architecture, normalization, optimization, view policy, and schedule—not these primitives in isolation.

## 6. Masked image modeling primitive

The encoder below receives visible patch embeddings only. Their context predicts every patch position through a lightweight decoder, and loss is computed only over masked patches. This is an intentionally small analogue of MAE's asymmetric path.

![Masked image modeling sends visible patches through an encoder and reconstructs hidden patches with a lightweight decoder.](assets/masked-image-modeling.svg)


In [ ]:
def patchify(images: torch.Tensor, patch_size: int = 8) -> torch.Tensor:
    batch, channels, height, width = images.shape
    if height % patch_size or width % patch_size:
        raise ValueError("height and width must be divisible by patch_size")
    patches = images.unfold(2, patch_size, patch_size).unfold(3, patch_size, patch_size)
    return patches.permute(0, 2, 3, 1, 4, 5).reshape(batch, -1, channels * patch_size * patch_size)


def unpatchify(patches: torch.Tensor, image_size: int = 64, patch_size: int = 8) -> torch.Tensor:
    batch, count, values = patches.shape
    grid = image_size // patch_size
    channels = values // (patch_size * patch_size)
    if count != grid * grid:
        raise ValueError("patch count does not match image_size and patch_size")
    return patches.reshape(batch, grid, grid, channels, patch_size, patch_size).permute(0, 3, 1, 4, 2, 5).reshape(batch, channels, image_size, image_size)


class TinyMaskedAutoencoder(nn.Module):
    def __init__(self, patch_dim: int = 192, tokens: int = 64, hidden: int = 48):
        super().__init__()
        self.position = nn.Parameter(torch.randn(tokens, hidden) * 0.02)
        self.encoder = nn.Sequential(nn.Linear(patch_dim, hidden), nn.GELU(), nn.LayerNorm(hidden))
        self.decoder = nn.Sequential(nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, patch_dim))

    def forward(self, patches: torch.Tensor, mask: torch.Tensor):
        encoded = self.encoder(patches) + self.position.unsqueeze(0)
        visible = (~mask).unsqueeze(-1)
        context = (encoded * visible).sum(dim=1) / visible.sum(dim=1).clamp_min(1)
        return self.decoder(context.unsqueeze(1) + self.position.unsqueeze(0))


tensor_transform = transforms.ToTensor()
mae_images = torch.stack([tensor_transform(samples[index].image) for index in train_indices[:96]])
mae_patches = patchify(mae_images)
torch.testing.assert_close(unpatchify(mae_patches), mae_images)
mae_model = TinyMaskedAutoencoder()
mae_optimizer = torch.optim.AdamW(mae_model.parameters(), lr=3e-3)
mask_generator = torch.Generator().manual_seed(SEED)
mae_losses = []
for step in range(60):
    batch_index = torch.randperm(len(mae_patches), generator=mask_generator)[:24]
    batch_patches = mae_patches[batch_index]
    noise = torch.rand(batch_patches.shape[:2], generator=mask_generator)
    mask = noise.argsort(dim=1) >= 16  # 48 of 64 patches masked = 75%
    prediction = mae_model(batch_patches, mask)
    loss = F.mse_loss(prediction[mask], batch_patches[mask])
    mae_optimizer.zero_grad(); loss.backward(); mae_optimizer.step()
    mae_losses.append(loss.item())

assert mask.float().mean().item() == 0.75
print({"tokens": 64, "visible_tokens": 16, "masked_tokens": 48, "first_loss": round(mae_losses[0], 4), "final_loss": round(mae_losses[-1], 4)})


In [ ]:
with torch.no_grad():
    source_patches = mae_patches[:1]
    demo_noise = torch.rand((1, 64), generator=torch.Generator().manual_seed(5))
    demo_mask = demo_noise.argsort(dim=1) >= 16
    predicted = mae_model(source_patches, demo_mask)
    masked_patches = source_patches.clone(); masked_patches[demo_mask] = 0
    reconstructed = source_patches.clone(); reconstructed[demo_mask] = predicted[demo_mask]
    masked_image = unpatchify(masked_patches)[0].permute(1, 2, 0).clamp(0, 1)
    reconstructed_image = unpatchify(reconstructed)[0].permute(1, 2, 0).clamp(0, 1)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
axes[0].imshow(mae_images[0].permute(1, 2, 0)); axes[0].set_title("original")
axes[1].imshow(masked_image); axes[1].set_title("75% masked")
axes[2].imshow(reconstructed_image); axes[2].set_title("tiny reconstruction")
axes[3].plot(mae_losses); axes[3].set_title("masked-only MSE"); axes[3].set_xlabel("step")
for ax in axes[:3]: ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()


The reconstruction becomes less wrong on its pretext objective. That does not establish semantic transfer. We now evaluate representations with held-out labels, source slices, and retrieval.

## 7. Extract random, supervised, and SSL representations

The random and SSL encoders share the same tiny architecture. The supervised baseline is official ImageNet-pretrained ResNet-18 from torchvision. Its upstream data, architecture, feature width, and compute differ, so this is a practical representation comparison—not an objective-only ablation.


In [ ]:
class ImageDataset(Dataset):
    def __init__(self, sample_rows, transform):
        self.sample_rows = list(sample_rows)
        self.transform = transform

    def __len__(self): return len(self.sample_rows)

    def __getitem__(self, index): return self.transform(self.sample_rows[index].image)


def extract_torch_features(model, transform, batch_size=64):
    loader = DataLoader(ImageDataset(samples, transform), batch_size=batch_size, shuffle=False, num_workers=0)
    result = []
    model.eval().to(DEVICE)
    with torch.inference_mode():
        for images in loader:
            result.append(model(images.to(DEVICE)).cpu().numpy())
    model.cpu()
    return np.concatenate(result)


evaluation_transform = transforms.ToTensor()
torch.manual_seed(SEED)
random_encoder = TinyEncoder().eval()
feature_store = {
    "Random tiny encoder": extract_torch_features(random_encoder, evaluation_transform),
    "SSL · weak views": extract_torch_features(ssl_encoders["weak"], evaluation_transform),
    "SSL · domain-valid views": extract_torch_features(ssl_encoders["domain_valid"], evaluation_transform),
    "SSL · domain-invalid views": extract_torch_features(ssl_encoders["domain_invalid"], evaluation_transform),
}

resnet_weights = ResNet18_Weights.DEFAULT
supervised_resnet = resnet18(weights=resnet_weights)
supervised_resnet.fc = nn.Identity()
feature_store["Supervised ImageNet · ResNet-18"] = extract_torch_features(supervised_resnet, resnet_weights.transforms(), batch_size=32)

print({name: values.shape for name, values in feature_store.items()})


## 8. Evaluate frozen geometry

We keep one source-held-out split for every representation:

- train/reference gallery: Factory A + Factory B;
- test/query set: Factory C;
- linear probe: standardized embeddings and logistic regression;
- k-NN: normalized embeddings, cosine-like Euclidean geometry;
- retrieval: precision@5 excluding self-matches by construction; and
- clustering: ARI and silhouette on the held-out source.

![A frozen encoder feeds global probe, nearest-neighbour, retrieval, geometry, and future patch-level evaluation.](assets/representation-evaluation.svg)


In [ ]:
labels = metadata.label.to_numpy()
sources = metadata.source.to_numpy()


def retrieval_precision_at_k(train_features, train_labels, query_features, query_labels, k=5):
    train_norm = normalize(train_features)
    query_norm = normalize(query_features)
    nearest = np.argsort(-(query_norm @ train_norm.T), axis=1)[:, :k]
    return float((train_labels[nearest] == query_labels[:, None]).mean())


def representation_metrics(features, name):
    x_train, x_test = features[train_indices], features[test_indices]
    y_train, y_test = labels[train_indices], labels[test_indices]
    probe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1500, random_state=SEED))
    probe.fit(x_train, y_train)
    probe_prediction = probe.predict(x_test)
    probe_f1 = f1_score(y_test, probe_prediction, average="macro")
    contamination_recall = recall_score(y_test == 3, probe_prediction == 3)
    knn = KNeighborsClassifier(n_neighbors=5, weights="distance")
    knn.fit(normalize(x_train), y_train)
    knn_f1 = f1_score(y_test, knn.predict(normalize(x_test)), average="macro")
    retrieval_p5 = retrieval_precision_at_k(x_train, y_train, x_test, y_test, k=5)
    test_norm = normalize(x_test)
    clusters = KMeans(n_clusters=len(CLASS_NAMES), n_init=10, random_state=SEED).fit_predict(test_norm)
    ari = adjusted_rand_score(y_test, clusters)
    silhouette = silhouette_score(test_norm, y_test, metric="cosine")
    return {"representation": name, "linear_probe_macro_f1": probe_f1, "contamination_recall": contamination_recall, "knn_macro_f1": knn_f1, "retrieval_precision_at_5": retrieval_p5, "class_ari": ari, "class_silhouette": silhouette}


evaluation_rows = [representation_metrics(values, name) for name, values in feature_store.items()]
evaluation_results = pd.DataFrame(evaluation_rows).sort_values("linear_probe_macro_f1", ascending=False)
evaluation_results.round(3)


### Measure collapse with several signals

Mean feature variance, pairwise cosine similarity, covariance, and effective rank answer different questions. We include an intentionally collapsed feature matrix as a reference failure rather than assigning a universal threshold.


In [ ]:
def collapse_diagnostics(features: np.ndarray, name: str):
    values = np.asarray(features, dtype=np.float64)
    centered = values - values.mean(axis=0, keepdims=True)
    covariance = np.cov(centered, rowvar=False)
    off_diagonal = covariance - np.diag(np.diag(covariance))
    singular_values = np.linalg.svd(centered, compute_uv=False)
    energy = singular_values / singular_values.sum().clip(min=1e-12)
    effective_rank = float(np.exp(-(energy * np.log(energy + 1e-12)).sum()))
    sampled = normalize(values[: min(200, len(values))])
    cosine = sampled @ sampled.T
    mean_pairwise = float(cosine[~np.eye(len(sampled), dtype=bool)].mean())
    return {
        "representation": name,
        "mean_feature_std": float(centered.std(axis=0).mean()),
        "mean_abs_offdiag_cov": float(np.abs(off_diagonal).mean()),
        "mean_pairwise_cosine": mean_pairwise,
        "effective_rank": effective_rank,
    }


collapsed_reference = np.ones((len(samples), 16))
collapse_rows = [collapse_diagnostics(collapsed_reference, "Deliberately collapsed reference")]
collapse_rows += [collapse_diagnostics(values, name) for name, values in feature_store.items()]
collapse_results = pd.DataFrame(collapse_rows)
collapse_results.round(4)


### Class separation versus source separation

A representation can look structured because it separates cameras rather than semantics. Cross-validated linear predictability provides one bounded comparison. High source predictability is not automatically bad—capture conditions are real—but it is a shortcut warning when source dominates the intended task.


In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
source_binary = (sources[train_indices] == "Factory B").astype(int)
separation_rows = []
for name, features in feature_store.items():
    values = features[train_indices]
    class_score = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), values, labels[train_indices], cv=cv, scoring="f1_macro").mean()
    source_score = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), values, source_binary, cv=cv, scoring="f1_macro").mean()
    separation_rows.append({"representation": name, "class_cv_f1": class_score, "source_cv_f1": source_score, "class_minus_source": class_score - source_score})
separation_results = pd.DataFrame(separation_rows).sort_values("class_minus_source", ascending=False)
separation_results.round(3)


In [ ]:
focus_name = "SSL · domain-valid views"
focus_features = normalize(feature_store[focus_name][test_indices])
projection = PCA(n_components=2, random_state=SEED).fit_transform(focus_features)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for label, class_name in enumerate(CLASS_NAMES):
    selected = labels[test_indices] == label
    axes[0].scatter(projection[selected, 0], projection[selected, 1], s=18, alpha=0.7, label=class_name)
axes[0].set_title("PCA coloured by class"); axes[0].legend(fontsize=7)
axes[1].scatter(projection[:, 0], projection[:, 1], c=[list(SOURCE_COLORS).index(source) for source in sources[test_indices]], s=18, alpha=0.7)
axes[1].set_title("Same PCA coloured by source")
for ax in axes: ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.grid(alpha=0.2)
plt.suptitle("A 2D projection is a diagnostic, not proof of representation quality")
plt.tight_layout(); plt.show()


## 9. Label-efficiency curves

For each class, the 1% budget selects one of its 100 training samples: five labelled examples in total. Every representation receives exactly the same deterministic subset at every fraction. The curve therefore asks how linearly accessible class structure is as labels increase.


In [ ]:
LABEL_FRACTIONS = [0.01, 0.05, 0.10, 0.25, 1.00]


def stratified_budget_indices(fraction: float):
    chosen = []
    rng = np.random.default_rng(SEED + int(fraction * 1000))
    for label in range(len(CLASS_NAMES)):
        candidates = train_indices[labels[train_indices] == label]
        count = max(1, int(round(len(candidates) * fraction)))
        chosen.extend(rng.choice(candidates, size=count, replace=False).tolist())
    return np.array(chosen)


label_efficiency_rows = []
for fraction in LABEL_FRACTIONS:
    selected = stratified_budget_indices(fraction)
    for name, features in feature_store.items():
        probe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1500, random_state=SEED))
        probe.fit(features[selected], labels[selected])
        prediction = probe.predict(features[test_indices])
        label_efficiency_rows.append({
            "fraction": fraction,
            "label_count": len(selected),
            "labels_per_class": len(selected) // len(CLASS_NAMES),
            "representation": name,
            "macro_f1": f1_score(labels[test_indices], prediction, average="macro"),
        })
label_efficiency = pd.DataFrame(label_efficiency_rows)
print(label_efficiency[["fraction", "label_count", "labels_per_class"]].drop_duplicates().to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
for name, group in label_efficiency.groupby("representation"):
    ax.plot(group.fraction * 100, group.macro_f1, marker="o", label=name)
ax.set(xlabel="available labels (%)", ylabel="held-out Factory C macro F1", title="Label efficiency under one controlled probe")
ax.set_xticks([1, 5, 10, 25, 100]); ax.set_ylim(0, 1.03); ax.grid(alpha=0.25); ax.legend(fontsize=7)
plt.show()
print(label_efficiency.pivot(index="representation", columns="fraction", values="macro_f1").round(3))


## 10. Global versus patch-level features

A pooled embedding is convenient for classification and retrieval. The final convolutional feature map contains an $8	imes8$ spatial grid. Flattening that grid creates 64 patch-like feature vectors. Pooling cannot recover spatial detail once it has been discarded.


In [ ]:
spatial_encoder = ssl_encoders["domain_valid"].eval()
example_tensor = evaluation_transform(samples[test_indices[35]].image).unsqueeze(0)
with torch.inference_mode():
    feature_map = spatial_encoder.forward_feature_map(example_tensor)
    patch_features = feature_map.flatten(2).transpose(1, 2)
    global_feature = spatial_encoder(example_tensor)
assert patch_features.shape == (1, 64, 64)
assert global_feature.shape == (1, 64)

patch_norms = patch_features.norm(dim=-1).reshape(8, 8).numpy()
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(samples[test_indices[35]].image); axes[0].set_title("input")
heatmap = axes[1].imshow(patch_norms, cmap="magma"); axes[1].set_title("8×8 local feature norms")
for ax in axes: ax.set_xticks([]); ax.set_yticks([])
plt.colorbar(heatmap, ax=axes[1], fraction=0.046); plt.tight_layout(); plt.show()
print({"global_embedding": tuple(global_feature.shape), "patch_features": tuple(patch_features.shape)})


## 11. Optional official DINOv2 extension

The default lab does not download or execute repository code through `torch.hub`. Set `CV_ENABLE_DINOV2=1` only after reviewing the official repository, model card, source revision, checkpoint license, preprocessing, memory, and network policy.

This extension retrieves both the global class token and patch tokens. It does not reproduce DINOv2 training and is excluded from the controlled local comparison unless the learner deliberately adds it to the same evaluation protocol.


In [ ]:
ENABLE_DINOV2 = os.getenv("CV_ENABLE_DINOV2", "0") == "1"
dinov2_summary = {"enabled": ENABLE_DINOV2, "repository": "facebookresearch/dinov2", "entrypoint": "dinov2_vits14"}
if ENABLE_DINOV2:
    dinov2 = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14", trust_repo=False).eval().to(DEVICE)
    dinov2_transform = transforms.Compose([
        transforms.Resize(256, antialias=True),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])
    dinov2_input = dinov2_transform(samples[test_indices[0]].image).unsqueeze(0).to(DEVICE)
    with torch.inference_mode():
        outputs = dinov2.forward_features(dinov2_input)
    dinov2_summary.update({
        "global_shape": list(outputs["x_norm_clstoken"].shape),
        "patch_shape": list(outputs["x_norm_patchtokens"].shape),
    })
else:
    dinov2_summary["status"] = "skipped by default; set CV_ENABLE_DINOV2=1 after reviewing source and weights"
print(json.dumps(dinov2_summary, indent=2))


## 12. Save evidence and make a provisional decision

Demonstration thresholds below are teaching gates for this generated corpus and notebook runtime only. They are not factory SLOs. The decision favors measured 10%-label performance, then checks collapse and source evidence; it does not infer a universal SSL winner.


In [ ]:
TEN_PERCENT_THRESHOLD_NOTICE = "Demonstration thresholds for this generated notebook corpus only."
candidate_names = [name for name in feature_store if name != "Random tiny encoder"]
ten_percent = label_efficiency[
    label_efficiency.fraction.eq(0.10) & label_efficiency.representation.isin(candidate_names)
].sort_values("macro_f1", ascending=False)
selected_representation = ten_percent.iloc[0].representation
selected_metrics = evaluation_results.set_index("representation").loc[selected_representation].to_dict()
selected_collapse = collapse_results.set_index("representation").loc[selected_representation].to_dict()
selected_separation = separation_results.set_index("representation").loc[selected_representation].to_dict()

evaluation_results.to_csv(ARTIFACT_DIR / "representation_evaluation.csv", index=False)
collapse_results.to_csv(ARTIFACT_DIR / "collapse_diagnostics.csv", index=False)
separation_results.to_csv(ARTIFACT_DIR / "class_source_separation.csv", index=False)
label_efficiency.to_csv(ARTIFACT_DIR / "label_efficiency.csv", index=False)
ssl_history.to_csv(ARTIFACT_DIR / "ssl_training_history.csv", index=False)

decision = {
    "course": "Beginner 04 — Self-Supervised Visual Representation Learning",
    "scenario": "bounded proxy for an industrial archive with scarce labels",
    "selected_representation": selected_representation,
    "selection_rule": "highest held-out-source macro F1 with 10% labelled training data among trained/pretrained candidates; random features are a baseline only",
    "threshold_notice": TEN_PERCENT_THRESHOLD_NOTICE,
    "selected_full_label_metrics": selected_metrics,
    "selected_collapse_diagnostics": selected_collapse,
    "selected_class_source_separation": selected_separation,
    "training_seconds_by_policy": training_times,
    "dinov2_extension": dinov2_summary,
    "risk_boundaries": [
        "synthetic data does not certify a production inspection system",
        "pretraining labels were hidden from SSL code but used for evaluation",
        "ImageNet supervision and tiny domain SSL differ in architecture, data, and compute",
        "2D projections and clustering are diagnostics, not semantic proof",
        "foundation-model claims are author-reported unless reproduced under this protocol",
    ],
    "production_next_steps": [
        "deduplicate and split by factory, camera, part, and time before pretraining",
        "review augmentation invariances with domain experts",
        "benchmark official frozen DINOv2/DINOv3 and simpler encoders under one contract",
        "evaluate dense patch features for detection and segmentation tasks",
        "pin code, checkpoints, preprocessing, licenses, and feature-index versions",
        "measure target-hardware cost and establish rollback and monitoring",
    ],
}
(ARTIFACT_DIR / "representation_decision.json").write_text(json.dumps(decision, indent=2), encoding="utf-8")
print(json.dumps(decision, indent=2))


## 13. What you should now be able to explain without code

1. Why is architecture different from a learning objective?
2. Why does an augmentation policy define invariance?
3. What does every row of the NT-Xent matrix represent?
4. Why does temperature change optimization pressure?
5. Why can low SSL loss coexist with weak downstream features?
6. What evidence indicates collapse?
7. Why do stop-gradient, predictor asymmetry, normalization, and EMA need to be considered together?
8. How do centering and sharpening change teacher distributions?
9. Why can masked reconstruction learn useful features without guaranteeing semantic quality?
10. What do linear probes, k-NN, retrieval, clustering, and label-efficiency curves each test?
11. Why can source separation masquerade as semantic structure?
12. Why do patch features matter for detection and segmentation?
13. When is domain SSL worth more than a strong existing frozen encoder?

## Production upgrade path

- replace procedural data with a governed, deduplicated corpus and immutable split manifest;
- train distributed objectives with pinned framework and checkpoint versions;
- validate mixed precision, queues/teachers, global batch semantics, and recovery from interrupted runs;
- log loss components, gradient norms, feature variance, effective rank, throughput, memory, and energy;
- compare frozen probes, adapters, and fine-tuning on source/time/rare-event slices;
- version preprocessing and embeddings; rebuild retrieval indexes atomically;
- document upstream data, licenses, source code, remote-code trust, and model limitations; and
- define review, rollback, retention, deletion, privacy, and incident-response controls.

**Next:** Course 05 turns reusable global and patch representations into spatial object predictions through localization, matching, and detection evaluation.
